# Day 17: CrewAI Deep Dive — Building a Content Crew

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> A content team is not one writer doing five jobs. It is five specialists, each with a distinct perspective.

Today we go deep on CrewAI's `role` / `goal` / `backstory` model by building a real
content production crew. The lesson is the **decomposition**: splitting "make a post"
into five specialist roles forces you to specify what each step actually requires.

## What You'll Build
- A 5-agent content crew: **Ideator → Hook Writer → Carousel Outliner → Caption Writer → Critic**
- Each task wired to its upstream with `context=[task]`
- One sequential `Crew` that outputs a full content package (angles → hooks → slides → caption → critique)


In [1]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 17 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 17 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Pipeline: Content Production Crew

```
Topic ──▶ Ideator ──▶ Hook Writer ──▶ Carousel Outliner ──▶ Caption Writer ──▶ Critic
        (3 angles)   (5 hooks ≤12w)   (6-8 slide titles)    (caption+CTA)    (SHIP/REVISE + fix)
```

Topic: **"Why every developer should learn AI agents in 2026"**

Five specialist agents. One sequential crew. The key design choice is what each agent
**does not** see — the Critic judges the shipped package, not the raw brainstorm, so
`ideation_task` is deliberately excluded from its `context`.


---
## CrewAI: Full Content Crew (Sequential)

Five agents, five tasks, one crew. In CrewAI's sequential process each task automatically
sees the previous task's output; `context=[task]` lets us pull in **non-adjacent** upstream
tasks (or make the contract explicit). Watch how the wiring below encodes "who sees what."


In [3]:
from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

# ── 1. Ideator ─────────────────────────────────────────────
ideator = Agent(
    role="Content Ideator",
    goal="Generate 3 distinct angles on the topic. Raw directions, no polish.",
    backstory=(
        "You are a strategist who has launched 500+ posts. You know the first idea is "
        "usually cliche, the second is safer, the third is where the insight lives. "
        "You output angles, not sentences."
    ),
    llm=llm, verbose=True,
)

# ── 2. Hook Writer ─────────────────────────────────────────
hook_writer = Agent(
    role="Hook Writer",
    goal="Turn the strongest angle into 5 scroll-stopping first lines, each under 12 words.",
    backstory=(
        "You study why posts stop a scroll: specificity, contradiction, a number, a cost "
        "named in the first 10 words. You write hooks like a copywriter, not a blogger. "
        "No setup sentence. No 'In today's world...'."
    ),
    llm=llm, verbose=True,
)

# ── 3. Carousel Outliner ───────────────────────────────────
carousel_outliner = Agent(
    role="Carousel Outliner",
    goal="Design 6-8 slide titles that teach the angle end-to-end.",
    backstory=(
        "You design swipeable carousels. Each slide title is one promise. Slide 1 is the hook "
        "restated visually, slides 2-6 are the payload, the last slide is the CTA. You think "
        "in visual structure, not paragraphs."
    ),
    llm=llm, verbose=True,
)

# ── 4. Caption Writer ──────────────────────────────────────
caption_writer = Agent(
    role="Caption Writer",
    goal="Write the caption: body, one specific CTA, and 3-5 hashtags. Match the hook's tone.",
    backstory=(
        "You write captions that earn the save, not just the like. Short sentences. One idea "
        "per paragraph. A CTA that asks for a specific action - not 'thoughts?'."
    ),
    llm=llm, verbose=True,
)

# ── 5. Critic ──────────────────────────────────────────────
critic = Agent(
    role="Senior Critic",
    goal="Roast the package. Output a SHIP or REVISE verdict with the one change that would move it from scannable to save-worthy.",
    backstory=(
        "10 years editing creators with millions of followers. You never say 'looks good'. "
        "You name the weakest element, the line a reader would screenshot, and the single "
        "change that would move the post from scannable to save-worthy."
    ),
    llm=llm, verbose=True,
)

# ── Tasks - each wired to its upstream via context=[...] ──
# In sequential process, each task auto-sees the previous task's output.
# context=[...] pulls in NON-ADJACENT upstream tasks (and makes the contract explicit).
ideation_task = Task(
    description=(
        "Topic: 'Why every developer should learn AI agents in 2026'. "
        "Generate 3 distinct angles. Each angle = one sentence + one reason it would land."
    ),
    expected_output="3 numbered angles, each with a one-line rationale.",
    agent=ideator,
)

hook_task = Task(
    description=(
        "Pick the strongest angle from the ideator. Write 5 hook variants for the first line. "
        "Each hook must be 12 words or fewer. No setup sentence."
    ),
    expected_output="5 numbered hooks.",
    agent=hook_writer,
    context=[ideation_task],  # explicit, though adjacent
)

carousel_task = Task(
    description=(
        "Design a 6-8 slide carousel outline for the chosen angle and hook. "
        "Output: slide number + slide title (one line each). Slide 1 = hook, last slide = CTA."
    ),
    expected_output="6-8 slide titles.",
    agent=carousel_outliner,
    context=[ideation_task, hook_task],  # ideation is non-adjacent (2 steps back)
)

caption_task = Task(
    description=(
        "Write the caption (120 words or fewer) for the post. Include body, one specific CTA, "
        "and 3-5 hashtags. Match the chosen hook's tone. Align the CTA with the carousel's last slide."
    ),
    expected_output="A caption with body, CTA, and hashtags.",
    agent=caption_writer,
    context=[hook_task, carousel_task],  # hook is non-adjacent
)

critic_task = Task(
    description=(
        "Review the full package: hooks, carousel outline, caption. "
        "Output: (1) weakest element and why, (2) the line a reader would screenshot, "
        "(3) the ONE change that would move this from scannable to save-worthy, "
        "(4) verdict: SHIP or REVISE."
    ),
    expected_output="A 4-point critique ending in SHIP or REVISE.",
    agent=critic,
    # Deliberately EXCLUDES ideation_task - the Critic judges the shipped package,
    # not the brainstorm. What you leave out of context is as important as what you put in.
    context=[hook_task, carousel_task, caption_task],
)

print("=" * 60)
print("CREWAI - CONTENT CREW (5 agents, sequential)")
print("=" * 60)

crew = Crew(
    agents=[ideator, hook_writer, carousel_outliner, caption_writer, critic],
    tasks=[ideation_task, hook_task, carousel_task, caption_task, critic_task],
    process=Process.sequential,
    verbose=True,
)

result = await crew.kickoff_async()
print(f"\n{'=' * 60}")
print("FINAL PACKAGE + CRITIQUE:")
print(f"{'=' * 60}")
print(result)


CREWAI - CONTENT CREW (5 agents, sequential)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4b041ce0-d726-41b7-8698-ec053022eb25                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Topic: 'Why every developer should learn AI agents in 2026'. Generate 3 distinct angles. Each angle =    │
│  one sentence + one reason it would land.                                                                       │
│  ID: 6be0e35b-1fce-4f46-ba48-82d9995ea34d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Ideator                                                                                         │
│                                                                                                                 │
│  Task: Topic: 'Why every developer should learn AI agents in 2026'. Generate 3 distinct angles. Each angle =    │
│  one sentence + one reason it would land.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Critic                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. Developers at the core of AI-driven innovation!                                                             │
│  2. Why learn AI? Futureproofing your career in software development.                                           │
│  3. The power of automation - how mastering AI can change your future.                                          │
│                                                                                                                 │
│  Weakest element:                                                                                               │
│  - The line "developers at the core" is slightly redundant given the focus on developers and innovation,        │
│  making it less impactful within the existing context.                                                          │
│                                                                                                                 │
│  Line a reader would screenshot:                                                                                │
│  - "Developers at the core of AI-driven innovation!" This sentence stands out due to its emphasis and could be  │
│  particularly eye-catching or controversial if not part of the overall flow.                                    │
│                                                                                                                 │
│  ONE change that would move this from scannable to save-worthy:                                                 │
│  - Replace "developers at the core" with "influential players guiding" for a subtle but more impactful shift    │
│  without losing directness.                                                                                     │
│                                                                                                                 │
│  VERDICT: SHIP                                                                                                  │
│                                                                                                                 │
│  This change maintains alignment with previous lines while adding a nuanced layer of meaning not captured by    │
│  redundant phrasing.                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the full package: hooks, carousel outline, caption. Output: (1) weakest element and why, (2) the  │
│  line a reader would screenshot, (3) the ONE change that would move this from scannable to save-worthy, (4)     │
│  verdict: SHIP or REVISE.                                                                                       │
│  Agent: Senior Critic                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4b041ce0-d726-41b7-8698-ec053022eb25                                                                       │
│  Final Output: 1. Developers at the core of AI-driven innovation!                                               │
│  2. Why learn AI? Futureproofing your career in software development.                                           │
│  3. The power of automation - how mastering AI can change your future.                                          │
│                                                                                                                 │
│  Weakest element:                                                                                               │
│  - The line "developers at the core" is slightly redundant given the focus on developers and innovation,        │
│  making it less impactful within the existing context.                                                          │
│                                                                                                                 │
│  Line a reader would screenshot:                                                                                │
│  - "Developers at the core of AI-driven innovation!" This sentence stands out due to its emphasis and could be  │
│  particularly eye-catching or controversial if not part of the overall flow.                                    │
│                                                                                                                 │
│  ONE change that would move this from scannable to save-worthy:                                                 │
│  - Replace "developers at the core" with "influential players guiding" for a subtle but more impactful shift    │
│  without losing directness.                                                                                     │
│                                                                                                                 │
│  VERDICT: SHIP                                                                                                  │
│                                                                                                                 │
│  This change maintains alignment with previous lines while adding a nuanced layer of meaning not captured by    │
│  redundant phrasing.                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL PACKAGE + CRITIQUE:
1. Developers at the core of AI-driven innovation!
2. Why learn AI? Futureproofing your career in software development.
3. The power of automation - how mastering AI can change your future.

Weakest element:
- The line "developers at the core" is slightly redundant given the focus on developers and innovation, making it less impactful within the existing context.

Line a reader would screenshot:
- "Developers at the core of AI-driven innovation!" This sentence stands out due to its emphasis and could be particularly eye-catching or controversial if not part of the overall flow.

ONE change that would move this from scannable to save-worthy:
- Replace "developers at the core" with "influential players guiding" for a subtle but more impactful shift without losing directness. 

VERDICT: SHIP

This change maintains alignment with previous lines while adding a nuanced layer of meaning not captured by redundant phrasing.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Why the Decomposition *Is* the Lesson

Three things make this crew work, and none of them are about the framework:

**1. Five roles, not five prompts.**
"Write me a post" produces cliche output because one prompt is doing five jobs. Splitting
ideation from hook-writing forces each agent to be specific about *what* it produces. The
Ideator outputs angles, not sentences. The Hook Writer outputs first lines under 12 words.
A different output shape per role = sharper output.

**2. Backstory shapes reasoning, not just output.**
"You write content" produces generic prose. "You know the first idea is cliche, the second
is safe, the third is where insight lives" makes the Ideator *reason differently* before it
writes. That is the CrewAI insight - backstory is a reasoning lens, not a label.

**3. `context=[task]` is the contract.**
Each task lists exactly which upstream outputs it can read. The Critic sees hooks, carousel,
caption - **not** the raw angles. That is deliberate: the Critic judges the *package*, not
the *process*. What you leave out of `context` is as important as what you put in. Getting
this wiring right is what separates a crew that works from five agents that happen to share
a topic.

> **Where this fits.** CrewAI is built for the "I need a team" mental model. The OpenAI SDK
> and LangGraph can both express the same pipeline - you'd hand-wire the context passing (SDK)
> or model it as graph edges (LangGraph). CrewAI's `context=[task]` is the shortest expression
> of the same wiring. We compare the three frameworks head-to-head on **Day 20**.


## Key Takeaway

CrewAI's sweet spot is multi-agent teams where **roles** matter. The `role` / `goal` /
`backstory` structure produces better agent behavior because it forces you to think about
**WHO** each agent is, not just **WHAT** it does. The content crew works not because CrewAI
is magic, but because five specialized perspectives beat one generalist prompt - every time.

For pipelines like this, `Process.sequential` is the right default. Reach for
`Process.hierarchical` only when you genuinely need a manager agent to decide who works next
(Day 14 covered the supervisor pattern; Day 18 covers Flows for deterministic steps around
an autonomous crew). To actually *act* on a REVISE verdict from the Critic, you would loop
the package back through the crew - that is Day 9 territory.

---

